# Hidden-size ablation on ACTION models — do the passive-GRU findings replicate?

**Sevan, 2026-08-13:** *"run an additional ablation of hidden state sizes but all on the action models
and make sure that our findings replicate or are proven to change."*

The plain-GRU hidden-size sweep (`../controls/hidden_size_sweep.ipynb`, `runs/controls/H{8…512}`)
produced four findings. This notebook re-runs the same ablation on **two action-model families** and
checks each one:

| # | finding on the passive GRU | tested in |
|---|---|---|
| **F1** | prediction saturates by `H ≈ 128` | §1 |
| **F2** | linear readability rises **monotonically** with `H` | §2 |
| **F3** | canonicality moves the **opposite** way (fiber residual grows) | §2 |
| **F4** | **grabbability does not move at all** — the §4 negative holds at every `H` | §3 |

**The two families were chosen because they differ on the axis that matters most here — whether the
model's own action space contains the intervention.**

* **Exogenous teleport** — a GRU conditioned on continuous **teleport-to-absolute-coordinate** actions.
  Its action space *contains the edit under test*: the model was trained to render exactly this
  intervention on command. So it ships with a **built-in ground-truth handle**, and "issue the action"
  is a positive control that no purely passive model can offer.
* **Endogenous interactive** — an actor that **acts on the world**: it emits forces, objects die on
  object–object and wall collisions, and it is trained with **REINFORCE on a survival reward into the
  same GRU trunk that is doing next-step prediction**. Its actions are forces, so it physically
  *cannot* teleport an object at any capacity — there is no action-interface oracle for this family,
  and that asymmetry is a property of the action space, not a gap in the experiment.

A third curve, the **passive GRU** (`runs/controls/H*`), is recomputed here with the *same* estimator
so all three are directly comparable — see the note in the definitions.

> ### ⚠ Scope — the TRAINED and FINE-TUNED editors are not here
> This notebook sweeps **hidden size** across 15 world models and evaluates **training-free** editors
> and oracles on each; training an editor per model would have been 90 extra runs. The trained
> mechanisms were therefore run at **H=256 only**, in the sibling notebook
> **[`../trained_editors_actions/trained_editors_actions.ipynb`](../trained_editors_actions/trained_editors_actions.ipynb)**:
> fine-tuning the world model (against both the pseudoinverse and the un-whitened write) and the
> **MLP editor `E(h, start, target)`**, each under a next-step and an 8-step rollout loss, on these
> same `XG_A_H256` / `XG_C_H256` models plus the standard-GRU control. **Every figure below shows
> training-free editors only** — nothing here is evidence about the trained ones.

**Registry:** `ACTION_SWEEP_RUNS.md` · **Training:** `scripts/train_action_hidden_sweep.sh` ·
**Evaluation:** `scripts/eval_action_sweep.py`

In [ ]:
# [1] setup
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

ROOT = Path("/home/sevan/research/physically-implicit-modeling")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/history_editing"))

from history_tools import ray_centroid, waterfall_grid   # the ONE waterfall spec, shared
from pim.figures.theme import style_ax

EVAL = ROOT / "runs/action_sweep/eval"
SIZES = [8, 32, 128, 256, 512]
FAMILIES = {
    "passive": ("Passive GRU (no actions)", "#5a5a5a", "o", ":"),
    "XG_C": ("Exogenous teleport · actions withheld", "#56B4E9", "s", "--"),
    "XG_A": ("Exogenous teleport · actions given", "#0072B2", "D", "-"),
    "EN": ("Endogenous L3 force+goal (RL + prediction)", "#009E73", "^", "-"),
}
OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
      "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#5a5a5a"}
print("eval dir:", EVAL, "|", len(list(EVAL.glob("*.json"))), "json files")

## Definitions

Every metric below is computed by `scripts/eval_action_sweep.py`, which routes the §4 editability
block through `scripts/editability_metrics.py` and readability through
`pim.extractors.fit_readability_probes`. Nothing is re-derived in this notebook.

### Runs (copied from `ACTION_SWEEP_RUNS.md` — the notebook stands alone)

| family | descriptive label | what it is | hidden sizes |
|---|---|---|---|
| `passive` | **Passive GRU (no actions)** | `runs/controls/H*` — the published hidden-size sweep. 400 epochs on `datasets/4_fixed_refl_inview`, pure next-step MSE, seed 0. The reference curve. | 8 … 512 |
| `XG_A` | **Exogenous teleport · actions given** | `ActionGRUContinuousModel` on `datasets/7_cont_teleport` (90k sequences, `p_action=0.30`, teleport to absolute coordinates). 400 epochs, batch 256, lr 1e-3, seed 0. | 8 … 512 |
| `XG_C` | **Exogenous teleport · actions withheld** | identical data and recipe, action input removed — isolates *action knowledge* from *capacity*. | 8 … 512 |
| `EN` | **Endogenous L3 force+goal** | `EndogenousActorGRU`, level 3: force dynamics, death on object/wall collision, REINFORCE + value baseline on survival (+0.1/step, −1.0/death) into the shared GRU trunk. 6000 iterations × batch 64 worlds × 48 frames, `--batched-sim`, seed 0. | 8 … 512 |

**Hidden size is the only variable within each family.** ⚠ The `EN_*` runs use `obs_noise 0.2` (the repo
standard); the older `runs/endogenous/L*` runs carry a known 0.05 deviation and are **not** directly
comparable to them.

### Metrics

| name | formula | units | better | notes |
|---|---|---|---|---|
| **next-step RMSE (vs clean)** | `RMSE(pred_{t+1}, clean_obs_{t+1})` over the probe split | obs intensity | ↓ | scored against the **clean** render, never the noisy observation. |
| **noise floor** | `RMSE(obs, clean_obs)` for that family's own data | obs intensity | — | a *reference scale* ("no better than echoing the input"), **not** a lower bound: a recurrent model that denoises many frames legitimately scores below it. |
| **position / velocity R²** | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, held out **by sequence**, vs the train mean | — | ↑ | linear = lstsq, MLP = 2×256 ReLU; both from `fit_readability_probes`. |
| **fiber residual** | `‖h − g(pos,vel)‖ / ‖h‖`, `g` linear or MLP | frac of ‖h‖ | ↓ (0 = canonical) | how much of `h` is *not* a function of the physical state. |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)` over **differing** rays, per sample then averaged | −1…+1 | ↑ | +1 = the edited world, −1 = the unedited one, **≈0 = equidistant OR garbage**. Read against **that model's own unsteered row** — a weaker predictor's unsteered value sits higher. |
| **Target / Ghost / Collateral RMSE** | `RMSE(edited₀, gt_edited)` over that zone at step 0 | obs intensity | ↓ | |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, gt_edited_s)` over the K=15 rollout | obs intensity | ↓ | |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ | **> 1 = the edit left the rollout further from the truth than doing nothing.** Load-bearing at small `H`, where editors degrade rather than steer. |

> ### ⚠ Why F2/F3 are **recomputed** for the passive family rather than read from its published JSON
> `scripts/eval_controls.py` splits its probe data **by row**, which leaks near-duplicate neighbouring
> frames, and uses a **1×128** MLP; this sweep uses `fit_readability_probes` (**2×256**, split by
> **sequence**). Those numbers are not comparable, and `CLAUDE.md` forbids quoting one probe as the
> other — so the passive family's F1/F2/F3 are recomputed here with the identical estimator. Its
> **editability** block *is* on the canonical §4 metric set and is read straight from
> `runs/controls/eval/*.json`.

### Editors

| editor | mechanism |
|---|---|
| **Unsteered** | no edit — the reference row for that model. |
| **Readout injection** | min-norm write to `h` so a linear position probe reads the target. |
| **Global-PCA projection** | the same write, alternating with projection onto the 99%-variance state subspace (POCS). |
| **MLP-probe gradient** | descend on `h` until a **frozen 1×128** `MLPExtractor` reads the target (its published defaults — a different object from the reporting probe). |
| **Decoder gradient (oracle)** | Adam on `h` so the decoder renders the true post-edit observation. Needs ground truth; the upper bracket. |
| **Action interface (oracle)** | **exogenous family only** — command the teleport through the model's own **action channel** instead of editing `h`. The built-in handle. The endogenous actor emits *forces* and has no such action, at any capacity. |

In [ ]:
# [2] load every evaluation
RUNS = {}
for fam in FAMILIES:
    for H in SIZES:
        code = f"passive_H{H}" if fam == "passive" else f"{fam}_H{H}"
        p = EVAL / f"{code}.json"
        if not p.exists():
            continue
        d = json.loads(p.read_text())
        if fam == "passive":     # editability comes from the published canonical-metric JSON
            pub = ROOT / f"runs/controls/eval/H{H}.json"
            if pub.exists():
                # keep the per-step curves — §4 plots `edit_index_by_step`, and stripping
                # the lists here is what silently dropped the passive column from Fig 4
                d["editability"] = json.loads(pub.read_text())["editability"]
        RUNS[(fam, H)] = d

have = sorted({f for f, _ in RUNS})
print(f"loaded {len(RUNS)} runs across families {have}")
for fam in FAMILIES:
    hs = sorted(H for f, H in RUNS if f == fam)
    print(f"  {FAMILIES[fam][0]:<45} H = {hs}")


# The published passive JSONs (`runs/controls/eval/`) use the short editor name.
ALIAS = {"Decoder gradient (oracle)": "Decoder gradient"}


def series(fam, key, default=np.nan):
    return [RUNS[(fam, H)].get(key, default) if (fam, H) in RUNS else np.nan for H in SIZES]


def edit_series(fam, editor, field="edit_index"):
    out = []
    for H in SIZES:
        ed = RUNS.get((fam, H), {}).get("editability", {})
        d = ed.get(editor) or ed.get(ALIAS.get(editor, editor))
        out.append(np.nan if d is None else d[field])
    return out

## §1 — F1: does prediction still saturate by H ≈ 128?

> ### ⚠ These predictive numbers are measured on a **teleport-free** world — read the action
> ### advantage accordingly
> The edit episodes are generated with `--p-action 0.0` so each carries exactly one intervention (see
> the definitions). That is required for the edit analysis, but it also means F1 is scored on a
> distribution where **nothing unpredictable ever happens** — which is precisely the thing the
> action-conditioned model is told about and the observer is not. Consequence: the observer's
> next-step RMSE at H=256 is **0.1234 here vs 0.1769 on the training distribution**, so the
> action-knowledge advantage shrinks from **0.070 to 0.017**. The *shape* of the F1 curve (where it
> flattens) is unaffected; the *size of the gap between the two exogenous families* is understated
> relative to the world these models were trained on. Measuring F1 on the training distribution while
> keeping the edits single-intervention is the clean complement, and is **not** done here.

In [ ]:
# [3] Fig 1 + Table 1 — predictive quality vs hidden size
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
for ax in axes:
    style_ax(ax)

ax = axes[0]
for fam, (lbl, col, mk, ls) in FAMILIES.items():
    if not any((fam, H) in RUNS for H in SIZES):
        continue
    ax.plot(SIZES, series(fam, "nextstep_rmse_vs_clean"), marker=mk, ls=ls, color=col, lw=2,
            ms=7, label=lbl)
    nf = np.nanmean(series(fam, "noise_floor_rmse"))
    # every reference line gets its own legend entry — an unlabelled dotted line is a bug
    ax.axhline(nf, color=col, lw=1, ls=":", alpha=0.7,
               label=f"    ↳ its noise floor = RMSE(obs, clean obs) = {nf:.3f}")
ax.set_xscale("log", base=2)
ax.set_xticks(SIZES)
ax.set_xticklabels(SIZES)
ax.set_xlabel("hidden size H")
ax.set_ylabel("next-step RMSE vs clean observations")
ax.set_title("(a) prediction — dotted line = that family's own noise floor", fontsize=10)
ax.legend(fontsize=7.5)

ax = axes[1]
for fam, (lbl, col, mk, ls) in FAMILIES.items():
    if not any((fam, H) in RUNS for H in SIZES):
        continue
    v = np.array(series(fam, "nextstep_rmse_vs_clean"), float)
    ax.plot(SIZES, v / np.nanmin(v), marker=mk, ls=ls, color=col, lw=2, ms=7, label=lbl)
ax.axhline(1.0, color="k", lw=0.9, label="that family's own best")
ax.set_xscale("log", base=2)
ax.set_xticks(SIZES)
ax.set_xticklabels(SIZES)
ax.set_xlabel("hidden size H")
ax.set_ylabel("next-step RMSE / that family's best")
ax.set_title("(b) the same curves normalised per family — where each one flattens", fontsize=10)
ax.legend(fontsize=7.5)

fig.suptitle("Fig 1 — F1: predictive quality vs hidden size, on action models "
             "(N=800 probe sequences)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

rows = ["| family | " + " | ".join(f"H={H}" for H in SIZES) + " | noise floor |",
        "|" + "---|" * (len(SIZES) + 2)]
for fam, (lbl, *_ ) in FAMILIES.items():
    if not any((fam, H) in RUNS for H in SIZES):
        continue
    v = np.array(series(fam, "nextstep_rmse_vs_clean"), float)
    rows.append(f"| {lbl} | " + " | ".join("—" if np.isnan(x) else f"{x:.4f}" for x in v)
                + f" | {np.nanmean(series(fam, 'noise_floor_rmse')):.4f} |")
display(Markdown("**Table 1 — next-step RMSE vs clean observations**, with each family's own noise "
                 "floor (= RMSE between the noisy observation and the clean render) as the reference "
                 "scale. The floor is a scale, **not** a bound: a recurrent model that denoises many "
                 "frames legitimately scores below it.\n\n" + "\n".join(rows)))

## §2 — F2 and F3: readability rises, canonicality falls?

In [ ]:
# [4] Fig 2 + Table 2 — recoverability and canonicality vs hidden size
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3))
for ax in axes:
    style_ax(ax)

# LATE-t (t >= 15) for both readouts: before then the belief has not converged, and an all-t
# velocity R² badly under-reads it (H256: all-t MLP 0.784 vs late-t 0.877). Registry rule.
panels = [("late_pos_r2_linear", "late_pos_r2_mlp", "(a) position R²  (late-t, t ≥ 15)", (-0.05, 1.02)),
          ("late_vel_r2_linear", "late_vel_r2_mlp", "(b) velocity R²  (late-t, t ≥ 15)", (-0.1, 1.02)),
          ("fiber_resid_linear", "fiber_resid_mlp", "(c) fiber residual (↓ = canonical)", (0, 1.02))]
for ax, (klin, kmlp, title, ylim) in zip(axes, panels):
    for fam, (lbl, col, mk, ls) in FAMILIES.items():
        if not any((fam, H) in RUNS for H in SIZES):
            continue
        ax.plot(SIZES, series(fam, klin), marker=mk, ls="-", color=col, lw=2, ms=6, label=f"{lbl} · linear")
        ax.plot(SIZES, series(fam, kmlp), marker=mk, ls="--", color=col, lw=1.4, ms=5, alpha=0.65)
    ax.set_xscale("log", base=2)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES)
    ax.set_xlabel("hidden size H")
    ax.set_title(title + "   (solid = linear, dashed = MLP)", fontsize=9.5)
    ax.set_ylim(*ylim)
axes[0].set_ylabel("held-out R²")
axes[2].set_ylabel("fraction of ‖h‖")
axes[0].legend(fontsize=7, loc="lower right")

fig.suptitle("Fig 2 — F2/F3: what the latent encodes vs hidden size  ·  readouts scored on "
             "LATE-t frames (t ≥ 15), where the belief has converged\n"
             "(standard probes: linear lstsq + 2×256 MLP, both held out by sequence)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

rows = ["| family | metric | " + " | ".join(f"H={H}" for H in SIZES) + " | monotone ↑? |",
        "|" + "---|" * (len(SIZES) + 3)]
for fam, (lbl, *_) in FAMILIES.items():
    if not any((fam, H) in RUNS for H in SIZES):
        continue
    for key, nice in (("late_pos_r2_linear", "position R² (linear, late-t)"),
                      ("late_vel_r2_linear", "velocity R² (linear, late-t)"),
                      ("late_vel_r2_mlp", "velocity R² (MLP, late-t)"),
                      ("fiber_resid_mlp", "fiber residual (MLP)")):
        v = np.array(series(fam, key), float)
        mono = "yes" if np.all(np.diff(v[~np.isnan(v)]) > -0.02) else "no"
        rows.append(f"| {lbl} | {nice} | " + " | ".join("—" if np.isnan(x) else f"{x:.3f}" for x in v)
                    + f" | **{mono}** |")
display(Markdown("**Table 2 — recoverability (F2) and canonicality (F3).** 'Monotone ↑?' allows a "
                 "0.02 tolerance for noise.\n\n" + "\n".join(rows)))

## §3 — F4: does grabbability move with capacity?

This is the finding that matters. On the passive GRU, capacity bought prediction and readability but
**no editability at any `H`**. The exogenous family adds the sharpest possible control: its **action
interface** performs the very same teleport through the channel the model was trained on, so if
anything in this figure moves with `H`, that arm should move first.

Every Edit Index is read against **its own model's unsteered row** — a weaker predictor's unsteered
value sits closer to 0 — and every claim is gated on the **fidelity ratio**, because at `H = 8`–`32`
these editors degrade the rollout rather than steer it.

In [ ]:
# [5] Fig 3 + Table 3 — editability vs hidden size
EDS = ["Unsteered", "Readout injection", "Global-PCA projection", "MLP-probe gradient",
       "Action interface (oracle)", "Decoder gradient (oracle)"]
ED_COL = {"Unsteered": OI["grey"], "Readout injection": OI["blue"],
          "Global-PCA projection": OI["sky"], "MLP-probe gradient": OI["purple"],
          "Action interface (oracle)": OI["orange"], "Decoder gradient (oracle)": OI["green"]}
fams_with_edits = [f for f in FAMILIES if any((f, H) in RUNS and "editability" in RUNS[(f, H)]
                                              for H in SIZES)]

fig, axes = plt.subplots(2, len(fams_with_edits), figsize=(4.6 * len(fams_with_edits), 8.2),
                         squeeze=False)
for j, fam in enumerate(fams_with_edits):
    ax = axes[0][j]
    style_ax(ax)
    for ed in EDS:
        v = edit_series(fam, ed)
        if np.all(np.isnan(v)):
            continue
        ax.plot(SIZES, v, marker="o", color=ED_COL[ed], lw=2, ms=6,
                ls="--" if "oracle" in ed else "-", label=ed)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xscale("log", base=2)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES)
    ax.set_ylim(-0.8, 1.02)
    ax.set_title(FAMILIES[fam][0], fontsize=9.5)
    if j == 0:
        ax.set_ylabel("Edit Index  (+1 = edited world, −1 = unedited)")

    ax = axes[1][j]
    style_ax(ax)
    for ed in EDS:
        v = edit_series(fam, ed, "fidelity_ratio")
        if np.all(np.isnan(v)):
            continue
        ax.plot(SIZES, v, marker="o", color=ED_COL[ed], lw=2, ms=6,
                ls="--" if "oracle" in ed else "-")
    ax.axhline(1.0, color=OI["red"], lw=1.4, ls="--")
    ax.set_xscale("log", base=2)
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES)
    ax.set_yscale("log")
    ax.set_xlabel("hidden size H")
    if j == 0:
        ax.set_ylabel("fidelity ratio (log)\n> 1 = degraded, not steered")

handles = [plt.Line2D([0], [0], color=ED_COL[e], lw=2.2, marker="o", ms=6,
                      ls="--" if "oracle" in e else "-", label=e) for e in EDS]
fig.legend(handles=handles, loc="upper center", ncol=len(EDS), fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, 0.955))
fig.suptitle("Fig 3 — F4: editability vs hidden size. Top = Edit Index (↑ better), "
             "bottom = the fidelity guard (↓ better, red line = doing nothing)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.925])
plt.show()

rows = ["| family | editor | " + " | ".join(f"H={H}" for H in SIZES)
        + " | gain vs own unsteered @ **H=512** | best legitimate gain (any H) |",
        "|" + "---|" * (len(SIZES) + 4)]
for fam in fams_with_edits:
    uns = np.array(edit_series(fam, "Unsteered"), float)
    for ed in EDS:
        v = np.array(edit_series(fam, ed), float)
        if np.all(np.isnan(v)):
            continue
        fid = np.array(edit_series(fam, ed, "fidelity_ratio"), float)
        legit = np.where(fid <= 1.05, v - uns, np.nan)
        best = "—" if np.all(np.isnan(legit)) else f"{np.nanmax(legit):+.3f}"
        big = "—" if np.isnan(legit[-1]) else f"**{legit[-1]:+.3f}**"
        cells = " | ".join("—" if np.isnan(x) else
                           (f"{x:+.3f}" + (" ⚠" if f > 1.05 else "")) for x, f in zip(v, fid))
        rows.append(f"| {FAMILIES[fam][0]} | {ed} | {cells} | {big} | {best} |")
display(Markdown(
    "**Table 3 — Edit Index by hidden size.** ⚠ = fidelity ratio > 1.05 (degraded, so that value is "
    "**not** an edit — at small `H` these editors wreck the rollout rather than steer it). "
    "The two right-hand columns are the gain over that model's own unsteered row: at the largest "
    "capacity, and the best across `H` **among arms that pass the fidelity guard**. Read the H=512 "
    "column for F4 — it is the one the trend points at.\n\n" + "\n".join(rows)))

## §4 — F4 in time: the Edit Index across the rollout

§3 scores the Edit Index at **step 0** — did the edit land on the frame it was aimed at. That says
nothing about whether the model then *keeps* the edited world, and `../METRICS_AND_EDITORS.md`
requires the by-step curve wherever the step-0 index is reported. Here it answers a sharper version
of F4: **does capacity change how an edit decays**, even where it does not change how it lands?

Scored against the counterfactual world **rolled forward** (the edited object continuing along its
own velocity, the other object on its true trajectory), so it stays bounded at every step — unlike an
RMSE against a static post-edit render.

Two editors, one per row: **readout injection**, the headline training-free write, and the
**decoder-gradient oracle**, the upper bracket. Colour = hidden size. In the top row each `H`'s own
**unsteered** curve is the faint dashed line of the same colour — the index has to be read against
it, and it climbs on its own as a free-running model drifts away from *both* reference worlds.

In [ ]:
# [6] Fig 4 — Edit Index across the K=15 rollout, by hidden size
def curve(fam, H, editor):
    d = RUNS.get((fam, H), {}).get("editability", {})
    e = d.get(editor) or d.get(ALIAS.get(editor, editor))
    c = None if e is None else e.get("edit_index_by_step")
    return None if not c else np.asarray(c)


fams_bs = [f for f in FAMILIES if any(curve(f, H, "Unsteered") is not None for H in SIZES)]
H_COL = {H: c for H, c in zip(SIZES, plt.cm.viridis(np.linspace(0.12, 0.88, len(SIZES))))}
ROWS = [("Readout injection", True), ("Decoder gradient (oracle)", False)]

fig, axes = plt.subplots(len(ROWS), len(fams_bs), figsize=(4.1 * len(fams_bs), 7.6),
                         squeeze=False, sharey=True, sharex=True)
for r, (editor, show_uns) in enumerate(ROWS):
    for j, fam in enumerate(fams_bs):
        ax = axes[r][j]
        style_ax(ax)
        ax.axhline(0, color="k", lw=0.9)
        for H in SIZES:
            v = curve(fam, H, editor)
            if v is None:
                continue
            ax.plot(np.arange(len(v)), v, color=H_COL[H], lw=2.0, marker="o", ms=2.6,
                    label=f"H={H}")
            if show_uns:
                u = curve(fam, H, "Unsteered")
                if u is not None:
                    # same colour as its solid partner and heavy enough to trace — a faint grey
                    # dotted line the reader cannot attribute to an H is worse than none
                    ax.plot(np.arange(len(u)), u, color=H_COL[H], lw=1.4, ls=":", alpha=0.95)
        ax.set_ylim(-1.02, 1.02)
        ax.set_xticks(np.arange(0, 15, 2))
        if r == 0:
            ax.set_title(FAMILIES[fam][0], fontsize=9)
        if r == len(ROWS) - 1:
            ax.set_xlabel("rollout step after the edit  (step 0 = the edit frame)")
        if j == 0:
            ax.set_ylabel(f"{editor}\nEdit Index (+1 = edited, −1 = unedited)", fontsize=9)
handles = ([plt.Line2D([0], [0], color=H_COL[H], lw=2.0, marker="o", ms=3, label=f"H={H}")
            for H in SIZES]
           + [plt.Line2D([0], [0], color="0.35", lw=1.4, ls=":", label="unsteered")])
axes[0][0].legend(handles=handles, fontsize=6.6, loc="upper left", ncol=2,
                  title="hidden size", title_fontsize=6.8)
fig.suptitle("Fig 4 — F4 in time: Edit Index across the rollout, by hidden size.\n"
             "Top row: solid = readout injection; dotted, in the same colour = that H's own "
             "unsteered curve, the reference the index must be read against.", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

rows = ["| family | editor | H | step 0 | step 4 | step 14 | gap to its own unsteered @ step 14 |",
        "|" + "---|" * 7]
for fam in fams_bs:
    for editor, _ in ROWS:
        for H in SIZES:
            v, u = curve(fam, H, editor), curve(fam, H, "Unsteered")
            if v is None or u is None:
                continue
            rows.append(f"| {FAMILIES[fam][0]} | {editor} | {H} | {v[0]:+.3f} | {v[4]:+.3f} | "
                        f"{v[-1]:+.3f} | **{v[-1] - u[-1]:+.3f}** |")
display(Markdown(
    "**Table 4 — Edit Index along the rollout.** The last column is the gap to that model's own "
    "unsteered curve at the final step: the unsteered index climbs on its own (a free-running model "
    "drifts from *both* reference worlds), so the raw value overstates persistence. A step-14 ÷ "
    "step-0 'retention' ratio is deliberately not reported — several arms have a step-0 index near "
    "zero, where that ratio explodes or flips sign.\n\n" + "\n".join(rows)))

## §5 — Observation space

`CLAUDE.md` requires any claim about an effect on the generations to ship with a waterfall, and this
is the first thing to read: whether an edit worked is visible immediately. Built through the same
`waterfall_grid(...)` helper as every other thread on this branch — gray on dark, a GT column of
clean sim observations, six **noisy** pre-edit context frames above the dashed edit line, and below
it every column is its own free-run from step 0.

Shown at the **smallest and largest** hidden size for each family — including the **passive GRU**,
which is the baseline everything else is read against — with **3 sample rows** each.

> ### The trained and fine-tuned editors are NOT in this notebook — they live in `../trained_editors_actions/`
> This sweep varies **hidden size** across 15 world models and evaluates **training-free** editors and
> oracles on each. Training an editor per model would have been 15 × 6 = 90 additional runs, so the
> trained mechanisms were run at **H=256 only**, in a sibling notebook:
> **`../trained_editors_actions/trained_editors_actions.ipynb`** — fine-tuning the world model
> (against both the pseudoinverse and the un-whitened write) and the **MLP editor
> `E(h, start, target)`**, each under a next-step and an 8-step rollout loss, on the same
> `XG_A_H256` / `XG_C_H256` models plus the standard-GRU control. **That is where the trained-editor
> waterfalls and rollout curves are.** Nothing here is evidence about them.

In [ ]:
# [7] Fig 5 — waterfalls at the extremes of the sweep (all four families, 3 rows each)
CTRL_EVAL = ROOT / "runs/controls/eval"


def draw_waterfall(fam, H, title):
    """The sweep families store `roll::<name>` + zone masks; the published passive runs store
    `roll_<name>` + precomputed centroids. One helper, both layouts."""
    if fam == "passive":
        p = CTRL_EVAL / f"H{H}_rollouts.npz"
        if not p.exists():
            print(f"(no rollouts for passive H{H})")
            return
        z = np.load(p)
        rolls = {k[len("roll_"):]: z[k] for k in z.files if k.startswith("roll_")}
        tgt_cx, ghost_cx = z["tgt_cx"], z["ghost_cx"]
        cards = RUNS[(fam, H)]["editability"]
    else:
        p = EVAL / f"{fam}_H{H}_rollouts.npz"
        if not p.exists():
            print(f"(no rollouts for {fam}_H{H})")
            return
        z = np.load(p)
        rolls = {k.split("::", 1)[1]: z[k] for k in z.files if k.startswith("roll::")}
        tgt_cx, ghost_cx = ray_centroid(z["tgt_mask"]), ray_centroid(z["ghost_mask"])
        cards = RUNS[(fam, H)]["editability"]

    order = [e for e in EDS if e in rolls] + [ALIAS[e] for e in EDS
                                              if e in ALIAS and ALIAS[e] in rolls and e not in rolls]
    labels = {}
    for e in order:
        c = cards.get(e) or cards.get(ALIAS.get(e, e))
        labels[e] = (f"{e}\nEdit Index {c['edit_index']:+.2f} · fid {c['fidelity_ratio']:.2f}"
                     if c else e)
    samples = list(np.argsort(z["teleport"])[::-1][:3])      # >= 3 rows, always
    fig = waterfall_grid(
        rolls={e: rolls[e] for e in order}, ctx=z["ctx"], gt_roll=z["gt_roll"],
        tgt_cx=tgt_cx, ghost_cx=ghost_cx, samples=samples, edit_frame=20,
        leads_by_one=("Oracle observation",), title=title, labels=labels)
    out = Path("/tmp/action_hidden_size")
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{fam}_H{H}_waterfall.png", dpi=120, bbox_inches="tight", facecolor="#0a0a14")
    plt.show()


for fam in FAMILIES:
    for H in (SIZES[0], SIZES[-1]):
        if (fam, H) in RUNS:
            draw_waterfall(fam, H,
                           f"Fig 5 — {FAMILIES[fam][0]} · {H} hidden — training-free editors and "
                           f"oracles\n(3 largest teleports; each column is that arm's own free-run. "
                           f"Trained/fine-tuned editors: see ../trained_editors_actions/)")

## Summary

**What this notebook measures** (invariant): whether the four findings of the plain-GRU hidden-size
sweep survive when the world model is action-conditioned — one family whose action space *contains*
the intervention (exogenous teleport), one that acts through *forces* under an RL survival objective
(endogenous), and the passive GRU recomputed with the identical estimator.
**Training-free editors and oracles only** — the trained and fine-tuned editors are in
`../trained_editors_actions/`.

### Current results (updated 2026-08-14)

**F1 — prediction saturates. REPLICATES in every family.** Next-step RMSE vs clean flattens by
`H ≈ 128`: passive 0.1499 → 0.1040, actions-given 0.1681 → 0.1071, actions-withheld 0.2016 → 0.1772,
endogenous 0.2080 → 0.1553. Action knowledge is worth a large constant (0.1071 vs 0.1769 at H=256 —
teleports are unpredictable without the action) but does not move *where* the curve flattens.

**F2 — readability rises with capacity. REPLICATES for position; velocity is noisier.** Late-t linear
position R²: passive 0.159 → 0.853, actions-given 0.196 → 0.781, actions-withheld 0.220 → 0.679, all
monotone. Velocity (MLP, late-t) rises but not monotonically in the exogenous families
(actions-given −0.049 → 0.367 at H=256 → 0.347 at H=512). **Endogenous breaks**: position R² rises
0.287 → 0.672 at H=256 then **collapses to 0.268 at H=512** — most likely under-training (every
endogenous run got the same 6000 iterations and the RL arm has the most to learn per parameter),
flagged not explained.

**F3 — canonicality moves the opposite way. REPLICATES everywhere.** MLP fiber residual rises with
capacity in all four families: passive 0.288 → 0.637, actions-given 0.410 → 0.695, actions-withheld
0.317 → 0.710, endogenous 0.270 → 0.500.

**F4 — grabbability. REPLICATES, and is STRONGER than "flat".** The legitimate gain of readout
injection over its own unsteered row shrinks toward zero as capacity grows, and §4 shows the curves
never separate from unsteered at **any** rollout step: the step-14 gap at `H ≥ 256` is **−0.002 … +0.052** across all
four families (and ≤ +0.007 at H=512) — indistinguishable from the unsteered curve it is supposed to
be steering away from. Every apparently large gain at `H = 8`–`32`
is degradation (fidelity 2.3–3.1; Fig 5's `H=8` panels show saturated bands, not a relocated object);
by `H ≥ 128` the editors are *inert* instead (fidelity 1.00, sitting on the unsteered line). The two
failure modes trade places as capacity grows and neither is an edit.

**The built-in handle is what makes it decisive.** Over exactly the range where latent editing decays
to nothing, the **action interface rises** — the model's own channel for this very teleport gets
better with capacity while the latent readout channel gets worse.

**Capacity does NOT buy edit persistence — a correction to an earlier version of this notebook.** An
earlier §4 read the *gap to unsteered* at step 14 and concluded larger models hold an edit better.
That was an artefact of the statistic. The **raw** step-14 index of the decoder-gradient oracle is
flat and near zero at every capacity and in every family:

| family | H=8 | H=32 | H=128 | H=256 | H=512 |
|---|---|---|---|---|---|
| Passive GRU | +0.228 | +0.089 | +0.106 | +0.023 | +0.165 |
| Exogenous · actions withheld | +0.045 | +0.088 | +0.089 | +0.124 | +0.141 |
| Exogenous · actions given | +0.141 | +0.135 | +0.084 | +0.078 | +0.070 |
| Endogenous L3 | +0.018 | −0.018 | +0.049 | −0.063 | −0.021 |

The oracle starts at +0.45…+0.99 and **reverts to ≈0 within 15 steps everywhere**, reproducing the
published decoder-gradient result (+0.94 → +0.08) and visible directly in Fig 5, where its column
jumps to the green target and then drifts back toward the ghost and breaks into streaks. The *gap* to
unsteered grows with `H` only because the **unsteered** index falls as the predictor improves.
*Methodological point:* the gap is the right statistic for "is this distinguishable from doing
nothing" across models with different unsteered rows, and the **wrong** one for "does the edit hold".
Table 4 reports both.

### Two measurement notes that matter for cross-notebook comparison

**Velocity R² here is much lower than in older notebooks, and this notebook's number is the honest
one.** Same 2×256 probe, same late-t window, only the split differs: **by sequence 0.565 vs by row
0.905** on `controls/H256`. Velocity is nearly constant within a sequence, so a row split hands the
probe a near-duplicate of the answer; position is far less affected (0.924 vs 0.971). Every
pre-2026-08-06 velocity number in this repo — including `runs/controls/eval`'s 0.877 and the
`findings/editability.md` 0.94 — is on the leaky convention and is **not** comparable to anything here.

**The reference futures are constructed, not read from the dataset.** `datasets/7_cont_teleport`
fires its own random teleports on ~30% of transitions; any landing inside the scored horizon would put
events into the ground truth that the free-running model was never told about. Both reference worlds
are therefore rolled forward from the frame-`ef` state under the passive (ballistic) dynamics, so the
scored window contains **exactly one intervention — the one under test**, matching every canonical
edits split. *(This was wrong in the first version of this notebook and every value has been recomputed.)*

### Owed / scope limits

* One seed per cell; 6000 iterations for every endogenous run regardless of `H` — the most likely
  cause of the `EN_H512` collapse, and it should be checked with a longer run before that dip is read
  as anything about the objective.
* Training-free editors only. The trained mechanisms exist at H=256 only, in
  `../trained_editors_actions/`; nothing here is evidence about them, and the capacity dependence of
  a *trained* editor is untested.
* The endogenous counterfactual is open-loop (the actor's own policy would diverge after a teleport).
* `EN_*` use `obs_noise 0.2` (repo standard) and so are not bit-comparable to the older
  `runs/endogenous/L*` runs, which carry a known 0.05 deviation.